In [ ]:
print("[cell] setup + ollama", flush=True)
import subprocess, os
def sh(cmd, check=True):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv || true", check=False)
sh("apt-get install -y -qq zstd > /dev/null 2>&1 || true", check=False)
sh("curl -fsSL https://ollama.com/install.sh | sh")
subprocess.Popen("ollama serve > /kaggle/working/ollama.log 2>&1", shell=True)
import time; time.sleep(8)
sh("ollama --version")


In [ ]:
print("[cell] write bundle", flush=True)
import os
B = {'src/quantum_gw/providers/base.py': 'from __future__ import annotations\n\nimport json\nimport re\nfrom abc import ABC, abstractmethod\nfrom typing import Any\n\nfrom pydantic import BaseModel, Field\n\n\nclass ProviderUnavailableError(RuntimeError):\n    """Raised when a provider cannot serve a request (not configured, offline, error)."""\n\n\nclass NoProviderAvailableError(RuntimeError):\n    """Raised when every provider in the routing chain is unavailable."""\n\n\nclass ModelInfo(BaseModel):\n    provider: str\n    model: str\n    endpoint: str | None = None\n    mode: str | None = None\n    extra: dict[str, Any] = Field(default_factory=dict)\n\n\nclass LLMResponse(BaseModel):\n    text: str\n    parsed: dict[str, Any] | None = None\n    provider: str\n    model: str\n    usage: dict[str, Any] = Field(default_factory=dict)\n\n\nclass LLMProvider(ABC):\n    """Common interface every model backend must implement.\n\n    Business modules must never talk to a concrete model API directly; they go\n    through this interface so switching local/FPT/cloud never touches pipeline logic.\n    """\n\n    name: str = "base"\n\n    @abstractmethod\n    def generate(\n        self, messages: list[dict[str, str]], response_schema: dict[str, Any] | None = None\n    ) -> LLMResponse: ...\n\n    def bind_model(self, model: str) -> LLMProvider:\n        """Return this provider aimed at a different model on the same endpoint.\n\n        One endpoint usually serves many models -- FPT Marketplace exposes a\n        dozen behind a single base URL -- and tasks have genuinely different\n        needs: claim extraction wants a fast small model, legal reasoning wants\n        the strongest one available. Without this, every task on a provider is\n        stuck with one model id.\n\n        Providers that cannot switch models return themselves unchanged, so\n        routing never fails because of an override it cannot honour.\n        """\n        return self\n\n    @abstractmethod\n    def healthcheck(self) -> bool: ...\n\n    @abstractmethod\n    def model_info(self) -> ModelInfo: ...\n\n    def is_configured(self) -> bool:\n        """True when required settings (endpoint/key/model) are present. Never does I/O."""\n        return True\n\n\ndef schema_instruction(response_schema: dict[str, Any]) -> str:\n    return (\n        "\\nReturn ONLY a JSON object matching this schema, no prose:\\n"\n        + json.dumps(response_schema, ensure_ascii=False)\n    )\n\n\ndef extract_json(content: str) -> dict[str, Any] | None:\n    match = re.search(r"\\{.*\\}", content, re.DOTALL)\n    if not match:\n        return None\n    try:\n        return json.loads(match.group(0))\n    except json.JSONDecodeError:\n        return None\n', 'src/quantum_gw/providers/ollama.py': 'from __future__ import annotations\n\nfrom typing import Any\n\nimport httpx\n\nfrom .base import (\n    LLMProvider,\n    LLMResponse,\n    ModelInfo,\n    ProviderUnavailableError,\n    extract_json,\n    schema_instruction,\n)\n\n\nclass OllamaProvider(LLMProvider):\n    """Native Ollama adapter for personal machines and development.\n\n    Ollama exposes a local API by default, so the repository can connect\n    without any cloud key (`LOCAL_LLM_PROVIDER=ollama`).\n    """\n\n    name = "ollama"\n\n    def __init__(self, base_url: str = "http://localhost:11434", model: str = "qwen3:8b", timeout: float = 180.0):\n        self.base_url = base_url.rstrip("/")\n        self.model = model\n        self.timeout = timeout\n\n    def is_configured(self) -> bool:\n        return bool(self.base_url and self.model)\n\n    def generate(\n        self, messages: list[dict[str, str]], response_schema: dict[str, Any] | None = None\n    ) -> LLMResponse:\n        if not self.is_configured():\n            raise ProviderUnavailableError("ollama: base URL/model not configured")\n        payload_messages = [dict(m) for m in messages]\n        if response_schema and payload_messages:\n            payload_messages[-1]["content"] += schema_instruction(response_schema)\n        payload: dict[str, Any] = {\n            "model": self.model,\n            "messages": payload_messages,\n            "stream": False,\n            "options": {"temperature": 0},\n        }\n        if response_schema:\n            payload["format"] = "json"\n        try:\n            with httpx.Client(timeout=self.timeout) as client:\n                response = client.post(f"{self.base_url}/api/chat", json=payload)\n                response.raise_for_status()\n                body = response.json()\n        except httpx.HTTPError as exc:\n            raise ProviderUnavailableError(f"ollama: {exc}") from exc\n        content = body["message"]["content"]\n        return LLMResponse(\n            text=content,\n            parsed=extract_json(content) if response_schema else None,\n            provider=self.name,\n            model=self.model,\n            usage={\n                "prompt_tokens": body.get("prompt_eval_count"),\n                "completion_tokens": body.get("eval_count"),\n            },\n        )\n\n    def healthcheck(self) -> bool:\n        if not self.is_configured():\n            return False\n        try:\n            with httpx.Client(timeout=5) as client:\n                return client.get(f"{self.base_url}/api/tags").status_code == 200\n        except httpx.HTTPError:\n            return False\n\n    def model_info(self) -> ModelInfo:\n        return ModelInfo(provider=self.name, model=self.model, endpoint=self.base_url)\n', 'src/quantum_gw/verification/ir_reader.py': '"""IR reader path behind QUANTUM_VERIFY=dual (registration agentic_ag2_product, gate ag2_dual_path_to_product).\n\nA model fills one IR per environmental figure of a sentence, with the AG-1 prompt unchanged\n(experiments/AG-1/ag1_worker.py; a test asserts they match). irlib.ground holds each reading to what the sentence\nwrites, and irlib.decide gives S / C / N by arithmetic in code (invariant 5). The path is a reviewer aid: it never\nchanges a verification\'s status. `dual` is the AG-2 v1 policy union_noloneS: a decisive label both paths give\nwithout conflict, except an S that only the rules path gives. Readings are cached per (model, sentence).\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport threading\nfrom pathlib import Path\n\nfrom quantum_gw.providers.base import LLMProvider\n\nfrom .irlib import IR, METRICS, SCOPES, VALUE_KINDS, decide, ground\n\nPROMPT = """Bạn trích số liệu môi trường từ MỘT câu trong báo cáo của doanh nghiệp Việt Nam.\nVới MỖI con số mô tả một chỉ số môi trường (không lấy số năm, số điều luật, số thứ tự, số tiền), trả về một đối tượng:\n- "metric": một mã trong {metrics}\n- "metric_text": cụm từ gọi tên chỉ số, chép NGUYÊN VĂN từ câu\n- "value_text": con số, chép NGUYÊN VĂN từ câu (giữ dấu chấm, phẩy), không kèm chữ\n- "multiplier_text": "nghìn", "triệu", "tỷ" nếu câu viết chữ đó ngay sau số, ngược lại ""\n- "unit_text": đơn vị nguyên văn (vd "tCO2e", "kWh", "m3", "tấn", "%", "tCO2e/tấn sản phẩm"); "" nếu không có\n- "value_kind": một trong {kinds} (share = tỷ lệ phần trăm của một tổng; change = mức tăng/giảm so với kỳ trước hoặc năm gốc; intensity = trên một đơn vị sản phẩm/doanh thu)\n- "change_direction": "decrease" | "increase" | null (chỉ khi value_kind = change)\n- "baseline_year": năm gốc nếu là change và câu có nêu, ngược lại null\n- "period": năm của số liệu nếu câu NÊU RÕ; null nếu câu không nêu\n- "status": "actual" (đã thực hiện) | "target" (mục tiêu, kế hoạch, cam kết)\n- "scope": một trong {scopes} nếu là phát thải khí nhà kính và câu nêu phạm vi, ngược lại null\n- "boundary": "group" (tập đoàn, hợp nhất) | "parent" (công ty mẹ) | "facility" (nhà máy, cơ sở cụ thể) | "unspecified"\nCâu không có số liệu môi trường thì trả về danh sách rỗng.\n\nCÂU: {sentence}\n\nChỉ trả lời đúng một đối tượng JSON: {{"figures": [ ... ]}}"""\n\n_DIGIT = re.compile(r"\\d")\n\n\ndef parse(text: str | None) -> dict | None:\n    """First JSON object in the reply; None when there is none."""\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth = 0\n        for i in range(start, len(text)):\n            depth += {"{": 1, "}": -1}.get(text[i], 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef to_irs(figures, sentence: str) -> list[IR]:\n    """Readable figures, each first held to what the sentence writes (irlib.ground)."""\n    out = []\n    for f in figures or []:\n        if isinstance(f, dict) and f.get("value_text") not in (None, ""):\n            if (f := ground(f, sentence)) is None:\n                continue\n            try:\n                ir = IR.from_dict({**f, "value_text": str(f["value_text"])})\n                ir.quantity()  # unreadable number -> dropped\n                out.append(ir)\n            except (TypeError, ValueError):\n                pass\n    return out\n\n\ndef verdict(claim_irs: list[IR], ev_irs: list[IR], ev_text: str) -> tuple[str, list[str]]:\n    """Any C -> C; every claim figure S -> S; else N."""\n    if not claim_irs:\n        return "N", ["không đọc được số liệu trong tuyên bố"]\n    labels, why = [], []\n    for c in claim_irs:\n        label, steps = decide(c, ev_irs, ev_text)\n        labels.append(label)\n        why.append(f"{c.value_text} {c.unit_text}: {label} " + "; ".join(s[1] for s in steps)[:300])\n    if "C" in labels:\n        return "C", why\n    return ("S" if all(x == "S" for x in labels) else "N"), why\n\n\ndef dual(rules_label: str, ir_label: str) -> str:\n    """AG-2 v1 union_noloneS over the rules path and one IR path."""\n    decisive = {lab for lab in (rules_label, ir_label) if lab in ("S", "C")}\n    if len(decisive) != 1:\n        return "N"\n    lab = decisive.pop()\n    return "N" if lab == "S" and ir_label != "S" else lab\n\n\nclass IRReader:\n    """Reads sentences into IR figures through one provider; replies are cached in a JSONL file."""\n\n    def __init__(self, provider: LLMProvider, cache_dir: str | Path | None = None):\n        self.provider = provider\n        self.model = getattr(provider, "model", provider.name)\n        self.cache_path = Path(cache_dir) / "ir_reader.jsonl" if cache_dir else None\n        self._cache: dict[str, str] = {}\n        self._lock = threading.Lock()\n        self.calls = 0\n        if self.cache_path and self.cache_path.exists():\n            for line in self.cache_path.open(encoding="utf-8"):\n                rec = json.loads(line)\n                self._cache[rec["key"]] = rec["raw"]\n\n    def _key(self, sentence: str) -> str:\n        return hashlib.sha1(f"{self.model}\\n{sentence}".encode()).hexdigest()\n\n    def raw(self, sentence: str) -> str:\n        """The model\'s reply for one sentence; a sentence without a digit is not sent (ground keeps no figure of it)."""\n        if not _DIGIT.search(sentence):\n            return \'{"figures": []}\'\n        key = self._key(sentence)\n        if key in self._cache:\n            return self._cache[key]\n        prompt = PROMPT.format(metrics=list(METRICS), kinds=list(VALUE_KINDS), scopes=list(SCOPES), sentence=sentence)\n        reply = self.provider.generate([{"role": "user", "content": prompt}]).text\n        with self._lock:\n            self.calls += 1\n            self._cache[key] = reply\n            if self.cache_path:\n                self.cache_path.parent.mkdir(parents=True, exist_ok=True)\n                with self.cache_path.open("a", encoding="utf-8") as fh:\n                    fh.write(json.dumps({"key": key, "model": self.model, "raw": reply}, ensure_ascii=False) + "\\n")\n        return reply\n\n    def read(self, sentence: str) -> list[IR]:\n        obj = parse(self.raw(sentence)) or {}\n        figures = obj.get("figures") if isinstance(obj.get("figures"), list) else []\n        return to_irs(figures, sentence)\n\n    def label(self, claim: str, evidence: str) -> tuple[str, list[str]]:\n        return verdict(self.read(claim), self.read(evidence), evidence)\n', 'src/quantum_gw/verification/irlib.py': '"""IR of one quantitative environmental figure + deterministic comparison (GreenScan AG-1, GS-Bench v1).\n\nA model may fill the IR; every number, unit, period, scope and boundary check is done here, so the\nverdict always points at an arithmetic step (invariants 4-6 of ARCHITECTURE_INVARIANTS.md):\n  - one dimension differs (metric, value kind, status, period, scope, boundary, unit) -> N, never C;\n  - a target never proves a result;\n  - values match within the precision each side was published to.\nPure stdlib so the same file runs on Kaggle and in the product.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import asdict, dataclass, field\n\nMETRICS = {\n    "ghg_emissions": "phát thải khí nhà kính",\n    "energy_consumption": "tiêu thụ năng lượng",\n    "electricity_consumption": "tiêu thụ điện",\n    "renewable_energy": "năng lượng tái tạo (sản xuất hoặc sử dụng)",\n    "fuel_consumption": "tiêu thụ nhiên liệu (than, dầu, khí, sinh khối)",\n    "water_use": "nước khai thác / sử dụng / tiêu thụ",\n    "wastewater": "nước thải phát sinh / xử lý / tuần hoàn",\n    "waste_generated": "chất thải phát sinh",\n    "waste_recycled": "chất thải tái chế / tái sử dụng / thu hồi",\n    "hazardous_waste": "chất thải nguy hại",\n    "recycled_input": "nguyên liệu tái chế đưa vào sản xuất",\n    "air_pollutants": "bụi, SO2, NOx, khí thải khác",\n    "land_trees": "trồng cây, diện tích xanh, rừng",\n    "other_env": "chỉ số môi trường khác",\n}\nVALUE_KINDS = ("absolute", "intensity", "share", "change", "count")\nSTATUSES = ("actual", "target")\nSCOPES = ("s1", "s2", "s3", "s1+s2", "s1+s2+s3")\nBOUNDARIES = ("group", "parent", "facility", "unspecified")\n\nMULTIPLIERS = {"nghìn": 1e3, "ngàn": 1e3, "ngàn tỷ": 1e12, "nghìn tỷ": 1e12, "triệu": 1e6, "tỷ": 1e9, "tỉ": 1e9,\n               "thousand": 1e3, "million": 1e6, "billion": 1e9, "": 1.0}\n\n# unit text -> (dimension, factor to the dimension\'s base unit)\n_UNITS = [\n    (r"(kt|nghìn tấn) ?co2", ("ghg_t", 1e3)), (r"kg ?co2", ("ghg_t", 1e-3)), (r"(t|tấn) ?co2", ("ghg_t", 1.0)),\n    (r"twh", ("energy_kwh", 1e9)), (r"gwh", ("energy_kwh", 1e6)), (r"mwh", ("energy_kwh", 1e3)), (r"kwh", ("energy_kwh", 1.0)),\n    (r"toe", ("energy_kwh", 11_630.0)), (r"\\btj\\b", ("energy_kwh", 277_777.8)), (r"\\bgj\\b", ("energy_kwh", 277.7778)),\n    (r"\\bmwp?\\b", ("power_mw", 1.0)), (r"\\bkwp?\\b", ("power_mw", 1e-3)),\n    (r"m3|m³|khối", ("volume_m3", 1.0)), (r"lít|\\bl\\b", ("volume_m3", 1e-3)),\n    (r"tấn|\\bt\\b|tons?|tonnes?", ("mass_t", 1.0)), (r"\\bkg\\b", ("mass_t", 1e-3)),\n    (r"%|phần trăm", ("percent", 1.0)), (r"\\bha\\b|hecta|héc", ("area_ha", 1.0)), (r"cây|trees?", ("count_tree", 1.0)),\n]\n\n\ndef unit_dimension(unit_text: str) -> tuple[str, float]:\n    """Dimension and factor of a unit; an intensity keeps its denominator (\'ghg_t/tấn thép\')."""\n    text = (unit_text or "").strip().lower()\n    num, _, den = text.partition("/")\n    dim, factor = next(((d, f) for pattern, (d, f) in _UNITS if re.search(pattern, num)), (num or "none", 1.0))\n    if den:\n        dim += "/" + re.sub(r"\\s+", " ", den.strip())\n    return dim, factor\n\n\ndef parse_number(text: str) -> tuple[float, float]:\n    """Value of a written number and half its last written digit (the published precision).\n\n    Vietnamese writes 22.540.603 and 23,5; some reports use 1,234.5. A separator followed by exactly\n    three digits is read as a thousands separator, any other as the decimal mark.\n    """\n    s = re.sub(r"\\s", "", text or "")\n    if not re.fullmatch(r"[\\d.,]+", s) or not re.search(r"\\d", s):\n        raise ValueError(f"not a number: {text!r}")\n    if "." in s and "," in s:\n        dec = "," if s.rfind(",") > s.rfind(".") else "."\n        integer, decimals = s.replace("." if dec == "," else ",", "").split(dec)\n    elif s.count(".") > 1 or s.count(",") > 1:\n        integer, decimals = re.sub(r"[.,]", "", s), ""\n    elif "." in s or "," in s:\n        integer, decimals = re.split(r"[.,]", s)\n        if len(decimals) == 3:\n            integer, decimals = integer + decimals, ""\n    else:\n        integer, decimals = s, ""\n    value = float(f"{integer or 0}.{decimals or 0}")\n    return value, 0.5 * 10 ** (-len(decimals))\n\n\n_NUM_RE = re.compile(r"(?<!\\d)\\d[\\d.,]*\\d|(?<![\\d.,])\\d")\n_MULT_WORDS = sorted((w for w in MULTIPLIERS if w), key=len, reverse=True)\n_YEAR_RE = re.compile(r"(?<!\\d)(?:19|20)\\d\\d(?!\\d)")\n\n\ndef _multiplier_at(text: str) -> str:\n    """Multiplier word that opens text (\'triệu tCO2e\' -> \'triệu\'), \'\' when none."""\n    text = text.lstrip().lower()\n    return next((w for w in _MULT_WORDS if re.match(re.escape(w) + r"(?!\\w)(?!\\s+lệ(?!\\w))", text)), "")  # not "tỷ lệ" (ratio)\n\n\ndef ground(fig: dict, sentence: str) -> dict | None:\n    """Hold a reader\'s figure to what the sentence writes (AG-1 error attribution, 31/10).\n    - number, multiplier and unit packed into value_text (\'23,5 triệu tCO2e\') are split;\n    - the number must be written in the sentence (same value; the sentence\'s own spelling is kept), else None;\n    - the multiplier is the one the sentence writes right after that number (\'\' when it writes none):\n      a reader adding \'nghìn\' to \'10.730 m3\' turned 11 of Qwen3-32B\'s verdicts wrong;\n    - period and baseline year must be years written in the sentence, else null.\n    """\n    value = str(fig.get("value_text") or "").strip()\n    m = _NUM_RE.search(value)\n    if not m:\n        return None\n    out = {**fig, "value_text": m.group(0)}\n    rest = value[m.end():].strip()\n    if rest:\n        mult = _multiplier_at(rest)\n        if mult and not (fig.get("multiplier_text") or "").strip():\n            out["multiplier_text"] = mult\n        rest = rest[len(mult):].strip() if mult else rest\n        if rest and not (fig.get("unit_text") or "").strip():\n            out["unit_text"] = rest\n    try:\n        want = parse_number(out["value_text"])[0]\n    except ValueError:\n        return None\n    hits = []\n    for t in _NUM_RE.finditer(sentence):\n        try:\n            if parse_number(t.group(0))[0] == want:\n                hits.append(t)\n        except ValueError:\n            pass\n    hits = [t for t in hits if t.group(0) == out["value_text"]] or hits\n    if not hits:\n        return None\n    out["value_text"] = hits[0].group(0)\n    written = {_multiplier_at(sentence[t.end():t.end() + 16]) for t in hits}\n    mult = (out.get("multiplier_text") or "").strip().lower()\n    if mult not in written:\n        if len(written) != 1:\n            return None\n        out["multiplier_text"] = written.pop()\n    years = {int(y) for y in _YEAR_RE.findall(sentence)}\n    for name in ("period", "baseline_year"):\n        v = out.get(name)\n        if v not in (None, "", "null") and (not str(v).isdigit() or int(v) not in years):\n            out[name] = None\n    return out\n\n\n@dataclass\nclass IR:\n    metric: str\n    value_text: str\n    unit_text: str = ""\n    multiplier_text: str = ""\n    value_kind: str = "absolute"\n    change_direction: str | None = None  # decrease | increase, for value_kind == change\n    baseline_year: int | None = None\n    period: int | None = None\n    status: str = "actual"\n    scope: str | None = None\n    boundary: str = "unspecified"\n    metric_text: str = ""\n    subject_text: str = ""\n    extra: dict = field(default_factory=dict)\n\n    @classmethod\n    def from_dict(cls, d: dict) -> IR:\n        known = {k: d.get(k) for k in cls.__dataclass_fields__ if k in d and k != "extra"}\n        ir = cls(**known)\n        ir.boundary = ir.boundary or "unspecified"\n        ir.status = ir.status or "actual"\n        ir.value_kind = ir.value_kind or "absolute"\n        for name in ("period", "baseline_year"):\n            v = getattr(ir, name)\n            setattr(ir, name, int(v) if v not in (None, "", "null") and str(v).isdigit() else None)\n        return ir\n\n    def as_dict(self) -> dict:\n        return asdict(self)\n\n    def quantity(self) -> tuple[float, float, str]:\n        """Value and tolerance in the dimension\'s base unit, and the dimension."""\n        value, half = parse_number(self.value_text)\n        mult = MULTIPLIERS.get((self.multiplier_text or "").strip().lower(), 1.0)\n        dim, factor = unit_dimension(self.unit_text)\n        return value * mult * factor, half * mult * factor, dim\n\n\n_KIND = {"count": "absolute"}  # v1.2: a count is an absolute figure; the unit dimension (cây, sự cố …) still tells them apart\n\n\ndef comparable(claim: IR, evidence: IR) -> tuple[bool, str]:\n    """Same metric, value kind, status, period, scope, boundary and unit dimension (invariant 4)."""\n    checks = [\n        ("chỉ số", claim.metric, evidence.metric),\n        ("loại giá trị", _KIND.get(claim.value_kind, claim.value_kind), _KIND.get(evidence.value_kind, evidence.value_kind)),\n        ("thực hiện / mục tiêu", claim.status, evidence.status),\n        ("kỳ", claim.period, evidence.period),\n        ("phạm vi phát thải", claim.scope, evidence.scope),\n    ]\n    if claim.value_kind == "change":\n        checks.append(("năm gốc", claim.baseline_year, evidence.baseline_year))\n    if "unspecified" not in (claim.boundary, evidence.boundary):\n        checks.append(("ranh giới tổ chức", claim.boundary, evidence.boundary))\n    for name, a, b in checks:\n        if a != b:\n            return False, f"khác {name}: {a} / {b}"\n    try:\n        dim_c, dim_e = claim.quantity()[2], evidence.quantity()[2]\n    except ValueError as exc:\n        return False, f"không đọc được số: {exc}"\n    if dim_c != dim_e:\n        return False, f"khác đơn vị: {dim_c} / {dim_e}"\n    return True, "so được"\n\n\ndef compare(claim: IR, evidence: IR) -> tuple[str, str]:\n    """S (khớp), C (mâu thuẫn) or N (không so được), with the arithmetic or the reason."""\n    ok, why = comparable(claim, evidence)\n    if not ok:\n        return "N", why\n    if claim.value_kind == "change" and claim.change_direction != evidence.change_direction:\n        return "C", f"ngược chiều: {claim.change_direction} / {evidence.change_direction}"\n    vc, tc, dim = claim.quantity()\n    ve, te, _ = evidence.quantity()\n    gap = abs(vc - ve)\n    label = "S" if gap <= tc + te else "C"\n    return label, f"|{vc:g} − {ve:g}| = {gap:g} {dim}; dung sai {tc + te:g}"\n\n\ndef written_in(ir: IR, text: str) -> bool:\n    """The text writes this figure\'s number with the same multiplier (whatever a reader extracted from it)."""\n    fig = ground({"value_text": ir.value_text, "multiplier_text": ir.multiplier_text}, text)\n    return fig is not None and (fig.get("multiplier_text") or "").lower() == (ir.multiplier_text or "").lower()\n\n\ndef decide(claim: IR, evidence: list[IR], evidence_text: str | None = None) -> tuple[str, list[tuple[str, str]]]:\n    """Verdict over every evidence figure: S if some comparable figure matches and none conflicts,\n    C if comparable figures only conflict, N otherwise (nothing comparable, or mixed: a person reads it).\n    v1.2: with the evidence text, a C that rests on a value gap becomes N when the text writes the claim\'s own\n    number and multiplier: the reader missed that figure (both false C of Qwen3-4B on held-out run D)."""\n    steps = [compare(claim, e) for e in evidence]\n    labels = {label for label, _ in steps if label != "N"}\n    label = labels.pop() if len(labels) == 1 else "N"\n    direction = any(w.startswith("ngược chiều") for _, w in steps)\n    if label == "C" and evidence_text is not None and not direction and written_in(claim, evidence_text):\n        steps.append(("N", "số của tuyên bố có trong câu bằng chứng nhưng chưa được đọc ra: chuyển người"))\n        label = "N"\n    return label, steps\n', 'experiments/AG-2/product_reader.py': '"""Product reader on a GS-Bench file (registration agentic_ag2_product): parity with a Kaggle run + reading time (CPU).\n\n    .venv/Scripts/python.exe -I experiments/AG-2/product_reader.py BENCH OUT_DIR [--model M] [--ref KAGGLE_CASES]\n\nReads every claim and evidence text through quantum_gw.verification.ir_reader (Ollama at localhost:11434), writes\nOUT_DIR/cases.jsonl (id, label, why; the format of ag1_worker / policies.py) and OUT_DIR/summary.json (seconds per\nmodel call, label agreement with --ref per case). Replies are cached in OUT_DIR, so a stopped run resumes.\n"""\nfrom __future__ import annotations\n\nimport json, sys, time\nfrom collections import Counter\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT / "src"))\nsys.stdout.reconfigure(encoding="utf-8")\nfrom quantum_gw.providers.ollama import OllamaProvider  # noqa: E402\nfrom quantum_gw.verification.ir_reader import IRReader  # noqa: E402\n\n\ndef main() -> None:\n    args = sys.argv[1:]\n    opt = {args[i]: args[i + 1] for i in range(len(args)) if args[i].startswith("--")}\n    bench, out = Path(args[0]), Path(args[1])\n    model = opt.get("--model", "qwen3:4b-instruct-2507-q4_K_M")\n    out.mkdir(parents=True, exist_ok=True)\n    cases = [json.loads(line) for line in bench.open(encoding="utf-8")]\n    reader = IRReader(OllamaProvider(model=model, timeout=600), out)\n    texts = list(dict.fromkeys(t for c in cases for t in (c["claim"], c["evidence"])))\n    print(f"[reader] {len(cases)} cases, {len(texts)} unique texts, model {model}", flush=True)\n    t0, spent = time.time(), []\n    for i, t in enumerate(texts, 1):\n        t1, before = time.time(), reader.calls\n        reader.raw(t)\n        if reader.calls > before:\n            spent.append(time.time() - t1)\n        if i % 10 == 0 or i == len(texts):\n            print(f"[reader] {i}/{len(texts)} texts, {len(spent)} calls, {sum(spent):.0f} s", flush=True)\n    labels = {}\n    with (out / "cases.jsonl").open("w", encoding="utf-8") as fh:\n        for c in cases:\n            label, why = reader.label(c["claim"], c["evidence"])\n            labels[c["id"]] = label\n            fh.write(json.dumps({"id": c["id"], "label": label, "why": why}, ensure_ascii=False) + "\\n")\n    summary = {"model": model, "cases": len(cases), "texts": len(texts), "calls": len(spent),\n               "s_per_call": round(sum(spent) / len(spent), 1) if spent else None, "wall_s": round(time.time() - t0),\n               "labels": dict(Counter(labels.values()))}\n    if "--ref" in opt:\n        ref = {r["id"]: r["label"] for r in map(json.loads, Path(opt["--ref"]).open(encoding="utf-8"))}\n        both = [k for k in labels if k in ref]\n        summary["ref"] = opt["--ref"]\n        summary["agreement"] = round(sum(labels[k] == ref[k] for k in both) / len(both), 3)\n        summary["confusion_ref_to_local"] = dict(Counter(f"{ref[k]}->{labels[k]}" for k in both if labels[k] != ref[k]))\n    (out / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")\n    print(f"[reader] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'benchmark/gs_bench_v1/gs_heldout_v12.jsonl': '{"id": "bh-256e9fb0b2610f5e-0-S0", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "S0 restate", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 300 kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-256e9fb0b2610f5e-0-C1", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C1 value", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 375 kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "375", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-256e9fb0b2610f5e-0-C2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C2 magnitude", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 300 nghìn kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-256e9fb0b2610f5e-0-N2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N2 status", "claim": "Công ty đặt mục tiêu tiết kiệm trung bình 300-1.000 kw điện đạt 300 kw vào năm 2030.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-256e9fb0b2610f5e-0-N5", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N5 basis", "claim": "Tiết kiệm trung bình 300-1.000 kw điện của Công ty là 300 kw/tấn sản phẩm.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-256e9fb0b2610f5e-0-N6", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N6 metric", "claim": "Lượng điện từ năng lượng tái tạo của Công ty là 300 kw.", "evidence": "Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "300", "unit_text": "kw", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết kiệm trung bình 300-1.000 kw điện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-86c0150d1bdb2265-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh của Công ty là 02 Đơn vị.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["S"], "claim_ir": {"metric": "land_trees", "value_text": "02", "unit_text": "Đơn vị", "multiplier_text": "", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "02", "unit_text": "Đơn vị", "multiplier_text": "", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-86c0150d1bdb2265-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh của Công ty là 7 Đơn vị.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "7", "unit_text": "Đơn vị", "multiplier_text": "", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "02", "unit_text": "Đơn vị", "multiplier_text": "", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-86c0150d1bdb2265-0-C2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C2 magnitude", "claim": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh của Công ty là 02 nghìn Đơn vị.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "02", "unit_text": "Đơn vị", "multiplier_text": "nghìn", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "02", "unit_text": "Đơn vị", "multiplier_text": "", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-86c0150d1bdb2265-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh đạt 02 Đơn vị vào năm 2030.", "evidence": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh 02 Đơn vị nhà máy và trang trại của Vinamilk được công nhận Trung hòa carbon theo tiêu chuẩn PAS 2060:2014 +200 triệu đô la Mỹ so với cùng kỳ (1)Theo Brand Finance", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "02", "unit_text": "Đơn vị", "multiplier_text": "", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "02", "unit_text": "Đơn vị", "multiplier_text": "", "value_kind": "count", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Diện tích rừng ngập mặn được Vinamilk khoanh nuôi tái sinh", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2924cd23d07ec79f-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năng suất của Công ty tăng 10% so với năm 2020.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "10", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2924cd23d07ec79f-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năng suất của Công ty tăng 25% so với năm 2020.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "25", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2924cd23d07ec79f-0-C3", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C3 direction", "claim": "Năng suất của Công ty giảm 10% so với năm 2020.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "10", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2924cd23d07ec79f-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu Năng suất tăng 10% vào năm 2030.", "evidence": "Mặc dù thời gian giãn cách kéo dài, PNJLAB cán đích với doanh thu tăng trưởng 17% và hàng hóa được giao đúng hẹn GIÁM ĐỊNH PNJ (PNJL) MỘT HÀNH TRÌNH BỨT PHÁ VƯƠN CAO NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN Năng suất tăng 10% so với năm 2020 Tăng 67% số lượng khách hàng Hàng hóa giao đúng hẹn trên 95%", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "10", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Năng suất", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-12d6bfbb4289bee1-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Thiệt hại của biến đổi khí hậu lên nền kinh tế của Công ty đạt 1,5 % GDP.", "evidence": "tế của nước ta trong thời gian qua do ô nhiễm môi trường gây ra tối thiểu từ 1,5% - 3% GDP Những thiệt hại của biến đổi khí hậu lên nền kinh tế của quốc gia gây ra rủi ro cho hoạt động kinh doanh của Traphaco xét ở khía cạnh thu nhập của người dân giảm và các ưu tên dành cho chăm sóc sức khỏe có thể được thay thế cho những nguy cơ khác cấp bách hơn của khách hàng nhằm phục vụ cho các mục têu ngắn hạn.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "1,5", "unit_text": "% GDP", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại của biến đổi khí hậu lên nền kinh tế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "1,5", "unit_text": "% GDP", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại của biến đổi khí hậu lên nền kinh tế", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-12d6bfbb4289bee1-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Thiệt hại của biến đổi khí hậu lên nền kinh tế của Công ty đạt 16,5 % GDP.", "evidence": "tế của nước ta trong thời gian qua do ô nhiễm môi trường gây ra tối thiểu từ 1,5% - 3% GDP Những thiệt hại của biến đổi khí hậu lên nền kinh tế của quốc gia gây ra rủi ro cho hoạt động kinh doanh của Traphaco xét ở khía cạnh thu nhập của người dân giảm và các ưu tên dành cho chăm sóc sức khỏe có thể được thay thế cho những nguy cơ khác cấp bách hơn của khách hàng nhằm phục vụ cho các mục têu ngắn hạn.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "16,5", "unit_text": "% GDP", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại của biến đổi khí hậu lên nền kinh tế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "1,5", "unit_text": "% GDP", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại của biến đổi khí hậu lên nền kinh tế", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-12d6bfbb4289bee1-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu thiệt hại của biến đổi khí hậu lên nền kinh tế đạt 1,5 % GDP vào năm 2030.", "evidence": "tế của nước ta trong thời gian qua do ô nhiễm môi trường gây ra tối thiểu từ 1,5% - 3% GDP Những thiệt hại của biến đổi khí hậu lên nền kinh tế của quốc gia gây ra rủi ro cho hoạt động kinh doanh của Traphaco xét ở khía cạnh thu nhập của người dân giảm và các ưu tên dành cho chăm sóc sức khỏe có thể được thay thế cho những nguy cơ khác cấp bách hơn của khách hàng nhằm phục vụ cho các mục têu ngắn hạn.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "1,5", "unit_text": "% GDP", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại của biến đổi khí hậu lên nền kinh tế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "1,5", "unit_text": "% GDP", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại của biến đổi khí hậu lên nền kinh tế", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-a37ff2f36e8c7a2b-0-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Năm 2025, cường độ phát thải khí nhà kính của Công ty đạt 0,03%.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["S"], "claim_ir": {"metric": "ghg_emissions", "value_text": "0,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "ghg_emissions", "value_text": "0,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-a37ff2f36e8c7a2b-0-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Năm 2025, cường độ phát thải khí nhà kính của Công ty đạt 15,03%.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["C"], "claim_ir": {"metric": "ghg_emissions", "value_text": "15,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "ghg_emissions", "value_text": "0,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-a37ff2f36e8c7a2b-0-N1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N1 period", "claim": "Năm 2024, cường độ phát thải khí nhà kính của Công ty đạt 0,03%.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["N"], "claim_ir": {"metric": "ghg_emissions", "value_text": "0,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2024, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "ghg_emissions", "value_text": "0,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-a37ff2f36e8c7a2b-0-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu cường độ phát thải khí nhà kính đạt 0,03% vào năm 2030.", "evidence": "Đáng chú ý, mức phát thải này chỉ tương đương khoảng 0,03% cường độ phát thải khí nhà kính trung bình của Việt Nam trong năm 2025 ước khoảng 56,52 tấn CO2/01 Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) Cường độ phát thải khí nhà kính tại Bảo Việt (tấn CO2/1 tỷ đồng doanh thu)", "labels": ["N"], "claim_ir": {"metric": "ghg_emissions", "value_text": "0,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "ghg_emissions", "value_text": "0,03", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cường độ phát thải khí nhà kính", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-42bb04afc41b5a1c-1-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2022, tiết giảm được giảm 12% của Công ty đạt 12%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-42bb04afc41b5a1c-1-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2022, tiết giảm được giảm 12% của Công ty đạt 27%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "27", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-42bb04afc41b5a1c-1-N1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N1 period", "claim": "Năm 2021, tiết giảm được giảm 12% của Công ty đạt 12%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-42bb04afc41b5a1c-1-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu tiết giảm được giảm 12% đạt 12% vào năm 2030.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-42bb04afc41b5a1c-1-N6", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N6 metric", "claim": "Năm 2022, lượng nước thải của Công ty đạt 12%.", "evidence": "Hình minh hoạ: sơ đồ R.O Quy trình giúp tiết giảm được giảm 12%, tương đương 4.000m3/năm khoảng 50 triệu một năm Giúp tiết kiệm chi phí vận hành Xử lý nước thải hiệu quả, thân thiện với môi trường Giảm phát thải ra môi trường BÁO CÁO THƯỜNG NIÊN 2022 TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM TỪ QUẢN TRỊ HIỆU QUẢ HÀNH TRÌNH BAY BỀN VỮNG VƯỢT THỬ THÁCH VƯƠN TẦM BÁO CÁO THƯỜNG NIÊN 2022", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiết giảm được giảm 12%", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-489f9d772e830f0e-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Diện tích của Công ty là 950 ha.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["S"], "claim_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-489f9d772e830f0e-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Diện tích của Công ty là 1.188 ha.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "1.188", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-489f9d772e830f0e-0-C2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C2 magnitude", "claim": "Diện tích của Công ty là 950 nghìn ha.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-489f9d772e830f0e-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu diện tích đạt 950 ha vào năm 2030.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-489f9d772e830f0e-0-N5", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N5 basis", "claim": "Diện tích của Công ty là 950 ha/tấn sản phẩm.", "evidence": "Hệ thống TTST Green Farm có diện tích gần 950 ha, trong đó có hơn 500 ha diện tích trồng trọt sản xuất nguyên liệu thức ăn thô xanh chất lượng cao bao gồm ngô, cỏ Mombasa, yến mạch, lúa,...", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "950", "unit_text": "ha", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "diện tích", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2abb39872ba73557-0-S0", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "S0 restate", "claim": "Thu hồi được của Công ty là 7.000 viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["S"], "claim_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2abb39872ba73557-0-C1", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C1 value", "claim": "Thu hồi được của Công ty là 4.900 viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["C"], "claim_ir": {"metric": "waste_recycled", "value_text": "4.900", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2abb39872ba73557-0-C2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C2 magnitude", "claim": "Thu hồi được của Công ty là 7.000 nghìn viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["C"], "claim_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2abb39872ba73557-0-N2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N2 status", "claim": "Công ty đặt mục tiêu thu hồi được đạt 7.000 viên pin cũ vào năm 2030.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["N"], "claim_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2abb39872ba73557-0-N5", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N5 basis", "claim": "Thu hồi được của Công ty là 7.000 viên pin cũ/tấn sản phẩm.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["N"], "claim_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2abb39872ba73557-0-N6", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N6 metric", "claim": "Lượng chất thải phát sinh của Công ty là 7.000 viên pin cũ.", "evidence": "Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "labels": ["N"], "claim_ir": {"metric": "waste_generated", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng chất thải phát sinh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "7.000", "unit_text": "viên pin cũ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thu hồi được", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-752eb0e07be3dea9-0-S0", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "S0 restate", "claim": "Công ty đặt mục tiêu cây xanh đạt 1 cây.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["S"], "claim_ir": {"metric": "land_trees", "value_text": "1", "unit_text": "cây", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "1", "unit_text": "cây", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-752eb0e07be3dea9-0-C1", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C1 value", "claim": "Công ty đặt mục tiêu cây xanh đạt 6 cây.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "6", "unit_text": "cây", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "1", "unit_text": "cây", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-752eb0e07be3dea9-0-N2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N2 status", "claim": "Cây xanh của Công ty là 1 cây.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "1", "unit_text": "cây", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "1", "unit_text": "cây", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-752eb0e07be3dea9-0-N5", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N5 basis", "claim": "Công ty đặt mục tiêu cây xanh đạt 1 cây/tấn sản phẩm.", "evidence": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "1", "unit_text": "cây/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "1", "unit_text": "cây", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cây xanh", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e4a22e9b25091836-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tỷ trọng vốn chủ sở hữu của Công ty đạt 67%.", "evidence": "Tỷ trọng nợ phải trả và nguồn vốn chủ sở hữu không có biến động nhiều trong các năm gần đây đặc »» Tài sản cố định/Tổng Tài sản »» Tài sản lưu động/Tổng Tài sản »» Nợ phải trả/Tổng nguồn vốn »» Nguồn vốn chủ sở hữu/Tổng nguồn vốn Biểu đồ cấu trúc tài sản - nguồn vốn (%) Nguồn vốn chủ sở hữu/ biệt tỷ trọng vốn chủ sở hữu luôn chiếm trên 67% tổng nguồn vốn thể hiện cơ cấu tài chính Công ty mang tính lành mạnh và bền vững.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "67", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ trọng vốn chủ sở hữu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "67", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ trọng vốn chủ sở hữu", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e4a22e9b25091836-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tỷ trọng vốn chủ sở hữu của Công ty đạt 82%.", "evidence": "Tỷ trọng nợ phải trả và nguồn vốn chủ sở hữu không có biến động nhiều trong các năm gần đây đặc »» Tài sản cố định/Tổng Tài sản »» Tài sản lưu động/Tổng Tài sản »» Nợ phải trả/Tổng nguồn vốn »» Nguồn vốn chủ sở hữu/Tổng nguồn vốn Biểu đồ cấu trúc tài sản - nguồn vốn (%) Nguồn vốn chủ sở hữu/ biệt tỷ trọng vốn chủ sở hữu luôn chiếm trên 67% tổng nguồn vốn thể hiện cơ cấu tài chính Công ty mang tính lành mạnh và bền vững.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "82", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ trọng vốn chủ sở hữu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "67", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ trọng vốn chủ sở hữu", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e4a22e9b25091836-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu tỷ trọng vốn chủ sở hữu đạt 67% vào năm 2030.", "evidence": "Tỷ trọng nợ phải trả và nguồn vốn chủ sở hữu không có biến động nhiều trong các năm gần đây đặc »» Tài sản cố định/Tổng Tài sản »» Tài sản lưu động/Tổng Tài sản »» Nợ phải trả/Tổng nguồn vốn »» Nguồn vốn chủ sở hữu/Tổng nguồn vốn Biểu đồ cấu trúc tài sản - nguồn vốn (%) Nguồn vốn chủ sở hữu/ biệt tỷ trọng vốn chủ sở hữu luôn chiếm trên 67% tổng nguồn vốn thể hiện cơ cấu tài chính Công ty mang tính lành mạnh và bền vững.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "67", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tỷ trọng vốn chủ sở hữu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "67", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ trọng vốn chủ sở hữu", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-079acf7a91109441-0-S0", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "S0 restate", "claim": "Rừng cao su của Công ty là 9 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["S"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-079acf7a91109441-0-C1", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "C1 value", "claim": "Rừng cao su của Công ty là 13 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "13", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-079acf7a91109441-0-C2", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "C2 magnitude", "claim": "Rừng cao su của Công ty là 9 triệu ha.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-079acf7a91109441-0-N2", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "N2 status", "claim": "Công ty đặt mục tiêu rừng cao su đạt 9 ngàn ha vào năm 2030.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-079acf7a91109441-0-N5", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "N5 basis", "claim": "Rừng cao su của Công ty là 9 ngàn ha/tấn sản phẩm.", "evidence": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha/tấn sản phẩm", "multiplier_text": "ngàn", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-cdea1d4e020ecf9e-0-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Năm 2025, thiệt hại về tài sản của Công ty là 119,9 nghìn tỷ đồng.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-cdea1d4e020ecf9e-0-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Năm 2025, thiệt hại về tài sản của Công ty là 95,9 nghìn tỷ đồng.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "95,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-cdea1d4e020ecf9e-0-N1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N1 period", "claim": "Năm 2024, thiệt hại về tài sản của Công ty là 119,9 nghìn tỷ đồng.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2024, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-cdea1d4e020ecf9e-0-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu thiệt hại về tài sản đạt 119,9 nghìn tỷ đồng vào năm 2030.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-cdea1d4e020ecf9e-0-N5", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N5 basis", "claim": "Năm 2025, thiệt hại về tài sản của Công ty là 119,9 nghìn tỷ đồng/tấn sản phẩm.", "evidence": "xuất ngày càng phức tạp, năm 2025, thiên tai làm 519 người chết, mất tích và bị thương trên phạm vi cả nước, thiệt hại về tài sản ước tính hơn 119,9 nghìn tỷ đồng tương đương khoảng 0,8% GDP, việc duy trì các giải pháp bảo vệ cho cây trồng, vật nuôi và mô hình nông nghiệp công nghệ cao tiếp tục có ý nghĩa lớn đối với khu vực Bảo hiểm nông nghiệp Nhóm sản phẩm bảo hiểm đại chúng 416: AN TOÀN SỨC KHỎE CỦA CỘNG ĐỒNG", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng/tấn sản phẩm", "multiplier_text": "nghìn tỷ", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "119,9", "unit_text": "đồng", "multiplier_text": "nghìn tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thiệt hại về tài sản", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-24ec31c88923203e-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "56% doanh thu của Công ty đạt 56%.", "evidence": "Hết năm 2012 tổng doanh thu của công ty đạt 1.477 tỷ đồng trong đó doanh thu các sản phẩm từ dược liệu của công ty đã đạt 56% doanh thu (tăng 28% so với năm 2011), những kết quả này là những minh chứng cho thấy hiệu quả của chiến lược sức khỏe xanh của TRAPHACO, đồng thời cũng khẳng định quan điểm phát triển bền vững của công ty: “Vì môi trường xanh, vì sức khỏe cộng đồng, vì sự phồn vinh của đất nước”.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "56", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "56% doanh thu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "56", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "56% doanh thu", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-24ec31c88923203e-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "56% doanh thu của Công ty đạt 71%.", "evidence": "Hết năm 2012 tổng doanh thu của công ty đạt 1.477 tỷ đồng trong đó doanh thu các sản phẩm từ dược liệu của công ty đã đạt 56% doanh thu (tăng 28% so với năm 2011), những kết quả này là những minh chứng cho thấy hiệu quả của chiến lược sức khỏe xanh của TRAPHACO, đồng thời cũng khẳng định quan điểm phát triển bền vững của công ty: “Vì môi trường xanh, vì sức khỏe cộng đồng, vì sự phồn vinh của đất nước”.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "71", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "56% doanh thu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "56", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "56% doanh thu", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-24ec31c88923203e-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu 56% doanh thu đạt 56% vào năm 2030.", "evidence": "Hết năm 2012 tổng doanh thu của công ty đạt 1.477 tỷ đồng trong đó doanh thu các sản phẩm từ dược liệu của công ty đã đạt 56% doanh thu (tăng 28% so với năm 2011), những kết quả này là những minh chứng cho thấy hiệu quả của chiến lược sức khỏe xanh của TRAPHACO, đồng thời cũng khẳng định quan điểm phát triển bền vững của công ty: “Vì môi trường xanh, vì sức khỏe cộng đồng, vì sự phồn vinh của đất nước”.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "56", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "56% doanh thu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "56", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "56% doanh thu", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Điện sản xuất của Công ty là 860.512 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-0-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Điện sản xuất của Công ty là 860,5 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "860,5", "unit_text": "KW", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Điện sản xuất của Công ty là 602.358 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "602.358", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Điện sản xuất của Công ty là 860.512 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Điện sản xuất đạt 860.512 KW vào năm 2030.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Điện sản xuất của Công ty là 860.512 KW/tấn sản phẩm.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng điện từ năng lượng tái tạo của Công ty là 860.512 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "860.512", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện sản xuất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-1-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Điện văn phòng của Công ty là 241.728 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-1-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Điện văn phòng của Công ty là 241,7 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "241,7", "unit_text": "KW", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-1-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Điện văn phòng của Công ty là 193.382 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "193.382", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-1-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Điện văn phòng của Công ty là 241.728 nghìn KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-1-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Điện văn phòng đạt 241.728 KW vào năm 2030.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-1-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Điện văn phòng của Công ty là 241.728 KW/tấn sản phẩm.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-7e4c135fa8258de0-1-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng điện từ năng lượng tái tạo của Công ty là 241.728 KW.", "evidence": "MÔI TRƯỜNG (Tiếp theo) SỬ DỤNG TÀI NGUYÊN NƯỚC VÀ TIÊU HAO NĂNG LƯỢNG TIẾT KIỆM - HIỆU QUẢ • Điện sản xuất: 860.512 KW • Điện văn phòng: 241.728 KW Doanh nghiệp chủ yếu sử dụng điện năng để phục vụ sản xuất và sinh hoạt, trong năm 2021 tồng mức tiêu thụ điện năng (KW) Tổng chi phí điện năm 2021 (VNĐ) Công ty sử dụng khí Gas và dầu DO phục vụ 2 mục đích chính đó là nấu ăn, làm các sản phẩm thử nghiệm và chạy máy phát điện khi có sự cố về điện hoặc thông báo mất điện từ cơ quan quản lý.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "241.728", "unit_text": "KW", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện văn phòng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2b222a6ddb74a7f1-0-S0", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "S0 restate", "claim": "Rừng cao su của Công ty là 9 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["S"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2b222a6ddb74a7f1-0-C1", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "C1 value", "claim": "Rừng cao su của Công ty là 14 ngàn ha.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "14", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2b222a6ddb74a7f1-0-C2", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "C2 magnitude", "claim": "Rừng cao su của Công ty là 9 triệu ha.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["C"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2b222a6ddb74a7f1-0-N2", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "N2 status", "claim": "Công ty đặt mục tiêu rừng cao su đạt 9 ngàn ha vào năm 2030.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-2b222a6ddb74a7f1-0-N5", "layer": "B-b", "company": "dpr", "sector": "cao su", "family": "N5 basis", "claim": "Rừng cao su của Công ty là 9 ngàn ha/tấn sản phẩm.", "evidence": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "labels": ["N"], "claim_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha/tấn sản phẩm", "multiplier_text": "ngàn", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "land_trees", "value_text": "9", "unit_text": "ha", "multiplier_text": "ngàn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "rừng cao su", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-aae28ea7d360fa29-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Năm 2022, chi phí điện năng của Công ty giảm 7,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-aae28ea7d360fa29-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Năm 2022, chi phí điện năng của Công ty giảm 22,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "22,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-aae28ea7d360fa29-0-C3", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C3 direction", "claim": "Năm 2022, chi phí điện năng của Công ty tăng 7,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-aae28ea7d360fa29-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu chi phí điện năng giảm 7,6% vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-aae28ea7d360fa29-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Năm 2022, lượng điện từ năng lượng tái tạo của Công ty giảm 7,6% so với năm 2022.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "7,6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2022, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chi phí điện năng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-aae28ea7d360fa29-1-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 2.760 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-1-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 3.864 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "3.864", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-1-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 2.760 nghìn kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-1-N1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N1 period", "claim": "Năm 2020, Khí gas phục vụ nấu nướng của Công ty là 2.760 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2020, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-1-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Khí gas phục vụ nấu nướng đạt 2.760 kg vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-1-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Năm 2021, Khí gas phục vụ nấu nướng của Công ty là 2.760 kg/tấn sản phẩm.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "2.760", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nấu nướng", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-2-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-2-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 1.069 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "1.069", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-2-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 nghìn kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-2-N1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N1 period", "claim": "Năm 2020, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 kg.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2020, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-2-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Khí gas phục vụ nghiên cứu sản phẩm đạt 855 kg vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-2-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Năm 2021, Khí gas phục vụ nghiên cứu sản phẩm của Công ty là 855 kg/tấn sản phẩm.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "855", "unit_text": "kg", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Khí gas phục vụ nghiên cứu sản phẩm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-4-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 100 lít.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-4-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 80 lít.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "80", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-4-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 100 nghìn lít.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-4-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng dầu DO chạy phục vụ máy phát điện đạt 100 lít vào năm 2030.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-aae28ea7d360fa29-4-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng dầu DO chạy phục vụ máy phát điện của Công ty là 100 lít/tấn sản phẩm.", "evidence": "So với năm 2022, chi phí điện năng giảm ~ 7,6% (năm 2021: 2.388.396.032 VNĐ) • Khí gas phục vụ nấu nướng: 2.760 kg • Khí gas phục vụ nghiên cứu sản phẩm: 855 kg • Tổng lượng tiêu thụ khí Gas trong năm 2021 (kg) • Tổng lượng dầu DO chạy phục vụ máy phát điện: ~ 100 lít • Tuyên truyền tiết kiệm điện, nước khi sử dụng cho người lao động bằng phương pháp đào tạo, pano áp phích.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "100", "unit_text": "lít", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng dầu DO chạy phục vụ máy phát điện", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-bbd4e51bc7042cbc-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tỉ trọng lớn nhất của Công ty đạt 83,8%.", "evidence": "Doanh thu hàng sản xuất là dòng hàng chiếm tỉ trọng lớn nhất 83,8% trong cơ cấu doanh thu của công ty, thể hiện sự phát triển bền vững của doanh nghiệp dược trong lĩnh vực sản xuất các mặt hàng thuốc đông dược và tân dược chất lượng cao nhằm chăm sóc và bảo vệ sức Bên cạnh việc tăng trưởng ở hàng sản xuất thì hàng khai thác cũng có mức tăng trưởng trên 54% so với cùng kỳ năm 2019.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "83,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỉ trọng lớn nhất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "83,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỉ trọng lớn nhất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-bbd4e51bc7042cbc-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tỉ trọng lớn nhất của Công ty đạt 68,8%.", "evidence": "Doanh thu hàng sản xuất là dòng hàng chiếm tỉ trọng lớn nhất 83,8% trong cơ cấu doanh thu của công ty, thể hiện sự phát triển bền vững của doanh nghiệp dược trong lĩnh vực sản xuất các mặt hàng thuốc đông dược và tân dược chất lượng cao nhằm chăm sóc và bảo vệ sức Bên cạnh việc tăng trưởng ở hàng sản xuất thì hàng khai thác cũng có mức tăng trưởng trên 54% so với cùng kỳ năm 2019.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "68,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỉ trọng lớn nhất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "83,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỉ trọng lớn nhất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-bbd4e51bc7042cbc-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu tỉ trọng lớn nhất đạt 83,8% vào năm 2030.", "evidence": "Doanh thu hàng sản xuất là dòng hàng chiếm tỉ trọng lớn nhất 83,8% trong cơ cấu doanh thu của công ty, thể hiện sự phát triển bền vững của doanh nghiệp dược trong lĩnh vực sản xuất các mặt hàng thuốc đông dược và tân dược chất lượng cao nhằm chăm sóc và bảo vệ sức Bên cạnh việc tăng trưởng ở hàng sản xuất thì hàng khai thác cũng có mức tăng trưởng trên 54% so với cùng kỳ năm 2019.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "83,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tỉ trọng lớn nhất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "83,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỉ trọng lớn nhất", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-bea6856b8ea1f475-0-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Năm 2025, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty tăng 3 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-bea6856b8ea1f475-0-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Năm 2025, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty tăng 8 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "8", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-bea6856b8ea1f475-0-C3", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C3 direction", "claim": "Năm 2025, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty giảm 3 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-bea6856b8ea1f475-0-N1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N1 period", "claim": "Năm 2024, cải thiện 3 bậc trong bảng điểm đánh giá của Công ty tăng 3 bậc so với năm 2023.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2024, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-bea6856b8ea1f475-0-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu cải thiện 3 bậc trong bảng điểm đánh giá tăng 3 bậc vào năm 2030.", "evidence": "Năm 2025, Bảo Việt đã cải thiện 3 bậc trong bảng điểm đánh giá so với năm 2023 và đạt điểm cao hơn 67% các doanh nghiệp khác trong nhóm ngành bảo hiểm (gồm 234 doanh nghiệp toàn cầu đủ điều kiện tham Những nỗ lực cải thiện trong năm của Bảo Việt bao gồm: Chiến lược giảm phát thải khí nhà kính, Quản trị rủi ro, Quản trị an toàn bảo mật trong công nghệ thông tin;", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "3", "unit_text": "bậc", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2023, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "cải thiện 3 bậc trong bảng điểm đánh giá", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-3c5c5f73d7747fec-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Hơn 4 triệu sản phẩm của Công ty là 4 triệu sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-3c5c5f73d7747fec-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Hơn 4 triệu sản phẩm của Công ty là 9 triệu sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "9", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-3c5c5f73d7747fec-0-C2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C2 magnitude", "claim": "Hơn 4 triệu sản phẩm của Công ty là 4 tỷ sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-3c5c5f73d7747fec-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu hơn 4 triệu sản phẩm đạt 4 triệu sản phẩm vào năm 2030.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-3c5c5f73d7747fec-0-N5", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N5 basis", "claim": "Hơn 4 triệu sản phẩm của Công ty là 4 triệu sản phẩm/tấn sản phẩm.", "evidence": "REFRESH • NHẤN NÚT TÁI TẠO - BỨT PHÁ VƯƠN XA CỦNG CỐ VÀ PHÁT TRIỂN CÁC NĂNG LỰC DOANH NGHIỆP Sở hữu dây chuyền sản xuất hiện đại, đội ngũ thiết kế 50 thành viên và lực lượng nghệ nhân, thợ kim hoàn hơn 1.000 người, hằng năm, PNJ tung ra thị trường hơn 4 triệu sản phẩm.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm/tấn sản phẩm", "multiplier_text": "triệu", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "4", "unit_text": "sản phẩm", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hơn 4 triệu sản phẩm", "subject_text": "", "extra": {}}, "fold": 1, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-ac0dcc6352bdadad-0-S0", "layer": "B-b", "company": "cng", "sector": "?", "family": "S0 restate", "claim": "Thị phần của Công ty đạt 70%.", "evidence": "“Được thành lập năm 2007, với tầm nhìn tiên phong trong sự nghiệp xanh hóa nguồn nhiên liệu tại Việt Nam, Công ty CP CNG Việt Nam hiện là đơn vị hàng đầu về cung cấp khí thiên nhiên nén (CNG – Compressed Natural Gas) và khí thiên nhiên hóa lỏng (LNG – Liquefied Natural Gas), chiếm lĩnh >70% thị phần và có đầy đủ cơ sở hạ tầng vận chuyển – phân phối trên khắp cả nước.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "70", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thị phần", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "70", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thị phần", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-ac0dcc6352bdadad-0-C1", "layer": "B-b", "company": "cng", "sector": "?", "family": "C1 value", "claim": "Thị phần của Công ty đạt 85%.", "evidence": "“Được thành lập năm 2007, với tầm nhìn tiên phong trong sự nghiệp xanh hóa nguồn nhiên liệu tại Việt Nam, Công ty CP CNG Việt Nam hiện là đơn vị hàng đầu về cung cấp khí thiên nhiên nén (CNG – Compressed Natural Gas) và khí thiên nhiên hóa lỏng (LNG – Liquefied Natural Gas), chiếm lĩnh >70% thị phần và có đầy đủ cơ sở hạ tầng vận chuyển – phân phối trên khắp cả nước.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thị phần", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "70", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thị phần", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-ac0dcc6352bdadad-0-N2", "layer": "B-b", "company": "cng", "sector": "?", "family": "N2 status", "claim": "Công ty đặt mục tiêu thị phần đạt 70% vào năm 2030.", "evidence": "“Được thành lập năm 2007, với tầm nhìn tiên phong trong sự nghiệp xanh hóa nguồn nhiên liệu tại Việt Nam, Công ty CP CNG Việt Nam hiện là đơn vị hàng đầu về cung cấp khí thiên nhiên nén (CNG – Compressed Natural Gas) và khí thiên nhiên hóa lỏng (LNG – Liquefied Natural Gas), chiếm lĩnh >70% thị phần và có đầy đủ cơ sở hạ tầng vận chuyển – phân phối trên khắp cả nước.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "70", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "thị phần", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "70", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thị phần", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-8be4715d7a95430c-0-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Năm 2025, tiêu thụ năng lượng điện của Công ty đạt 35%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-8be4715d7a95430c-0-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Năm 2025, tiêu thụ năng lượng điện của Công ty đạt 50%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "50", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-8be4715d7a95430c-0-N1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N1 period", "claim": "Năm 2024, tiêu thụ năng lượng điện của Công ty đạt 35%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2024, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-8be4715d7a95430c-0-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu tiêu thụ năng lượng điện đạt 35% vào năm 2030.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-8be4715d7a95430c-0-N6", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N6 metric", "claim": "Năm 2025, lượng điện từ năng lượng tái tạo của Công ty đạt 35%.", "evidence": "Theo số liệu Báo cáo tóm tắt về sử dụng năng lượng hợp lý tại các tòa nhà của Hội đồng Doanh nghiệp vì sự phát triển bền vững thế giới, trong năm 2025, nhóm các tòa nhà văn phòng, xưởng và nhà máy tiếp tục là một trong những nhóm tiêu thụ năng lượng điện lớn nhất, chiếm khoảng 35%–40% tổng năng lượng toàn cầu, đóng góp đáng kể vào phát thải khí nhà kính.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "35", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ năng lượng điện", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-de130211938fb92a-0-S0", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "S0 restate", "claim": "Tạo ra việc làm cho của Công ty là 2.300 nhân viên.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-de130211938fb92a-0-C1", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C1 value", "claim": "Tạo ra việc làm cho của Công ty là 1.840 nhân viên.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "1.840", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-de130211938fb92a-0-C2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C2 magnitude", "claim": "Tạo ra việc làm cho của Công ty là 2.300 nghìn nhân viên.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-de130211938fb92a-0-N2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tạo ra việc làm cho đạt 2.300 nhân viên vào năm 2030.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-de130211938fb92a-0-N5", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N5 basis", "claim": "Tạo ra việc làm cho của Công ty là 2.300 nhân viên/tấn sản phẩm.", "evidence": "Giảm thiểu các ảnh • Phát triển cơ sở hạ • Tạo việc làm cho sinh viên và thu hút tài năng • Trên 850 dự án trải dài trên khắp Việt Nam và các nước trong khu vực Lào, • Tạo ra việc làm cho trên 2.300 nhân viên cùng hơn 30.000 lực lượng thi công • Thành lập Quỹ chung một tấm lòng.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "2.300", "unit_text": "nhân viên", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tạo ra việc làm cho", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-0-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Chiếm tỷ trọng của Công ty đạt 39,2%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "39,2", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "39,2", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-0-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Chiếm tỷ trọng của Công ty đạt 54,2%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "54,2", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "39,2", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-0-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu Chiếm tỷ trọng đạt 39,2% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "39,2", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "39,2", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-1-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Chiếm tỷ trọng của Công ty đạt 43,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "43,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "43,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-1-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Chiếm tỷ trọng của Công ty đạt 58,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "58,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "43,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-1-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu Chiếm tỷ trọng đạt 43,6% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "43,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "43,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-2-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Chiếm tỷ trọng của Công ty đạt 14,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "14,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "14,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-2-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Chiếm tỷ trọng của Công ty đạt 29,6%.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "29,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "14,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-2-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu Chiếm tỷ trọng đạt 14,6% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "14,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "14,6", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Chiếm tỷ trọng", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-3-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Năm 2025, đóng góp \uf0f110,9% của Công ty tăng 10,9% so với năm 2024.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "10,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-3-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Năm 2025, đóng góp \uf0f110,9% của Công ty tăng 25,9% so với năm 2024.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "25,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-3-C3", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C3 direction", "claim": "Năm 2025, đóng góp \uf0f110,9% của Công ty giảm 10,9% so với năm 2024.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "10,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e3f000701dd57fce-3-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu đóng góp \uf0f110,9% tăng 10,9% vào năm 2030.", "evidence": "Tri ân anh hùng, gia đình liệt sỹ có công với đất nước thiên tai và môi trường Chiếm tỷ trọng 39,2% Chiếm tỷ trọng 43,6% Chiếm tỷ trọng 14,6% CHỈ TIÊU CỘNG ĐỒNG ĐỊA PHƯƠNG ĐƯỢC ĐẢM BẢO BỞI KIỂM TOÁN NỘI BỘ TẬP ĐOÀN BẢO VIỆT cho sự phát triển bền vững của cộng đồng địa phương thông qua việc ưu tiên đầu tư cho các dự án thuộc phạm vi: Năm 2025, Bảo Việt đóng góp \uf0f110,9% so với năm 2024 ĐÓNG GÓP CHO SỰ PHÁT TRIỂN CỦA CỘNG ĐỒNG BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "10,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "10,9", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "đóng góp \uf0f110,9%", "subject_text": "", "extra": {}}, "fold": 4, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e771363ec78bac11-0-S0", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "S0 restate", "claim": "Tổng giờ thực hiện của Công ty là 651 giờ.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e771363ec78bac11-0-C1", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C1 value", "claim": "Tổng giờ thực hiện của Công ty là 814 giờ.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "814", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e771363ec78bac11-0-C2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C2 magnitude", "claim": "Tổng giờ thực hiện của Công ty là 651 nghìn giờ.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e771363ec78bac11-0-N2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N2 status", "claim": "Công ty đặt mục tiêu tổng giờ thực hiện đạt 651 giờ vào năm 2030.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-e771363ec78bac11-0-N5", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N5 basis", "claim": "Tổng giờ thực hiện của Công ty là 651 giờ/tấn sản phẩm.", "evidence": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "651", "unit_text": "giờ", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng giờ thực hiện", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-1c6744cbbe0ddbef-0-S0", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "S0 restate", "claim": "Chai nhựa tái chế của Công ty là 5 chai.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["S"], "claim_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-1c6744cbbe0ddbef-0-C1", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C1 value", "claim": "Chai nhựa tái chế của Công ty là 10 chai.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["C"], "claim_ir": {"metric": "recycled_input", "value_text": "10", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-1c6744cbbe0ddbef-0-C2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "C2 magnitude", "claim": "Chai nhựa tái chế của Công ty là 5 nghìn chai.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["C"], "claim_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-1c6744cbbe0ddbef-0-N2", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N2 status", "claim": "Công ty đặt mục tiêu chai nhựa tái chế đạt 5 chai vào năm 2030.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["N"], "claim_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-1c6744cbbe0ddbef-0-N5", "layer": "B-b", "company": "ctd", "sector": "xây dựng", "family": "N5 basis", "claim": "Chai nhựa tái chế của Công ty là 5 chai/tấn sản phẩm.", "evidence": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "labels": ["N"], "claim_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "recycled_input", "value_text": "5", "unit_text": "chai", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "chai nhựa tái chế", "subject_text": "", "extra": {}}, "fold": 3, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-4eaa023bef040348-0-S0", "layer": "B-b", "company": "pow", "sector": "điện", "family": "S0 restate", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-4eaa023bef040348-0-S1", "layer": "B-b", "company": "pow", "sector": "điện", "family": "S1 rounded", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3,0 triệu tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "3,0", "unit_text": "tấn", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-4eaa023bef040348-0-C1", "layer": "B-b", "company": "pow", "sector": "điện", "family": "C1 value", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 2.400.000 tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "2.400.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-4eaa023bef040348-0-C2", "layer": "B-b", "company": "pow", "sector": "điện", "family": "C2 magnitude", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 nghìn tấn vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-4eaa023bef040348-0-N1", "layer": "B-b", "company": "pow", "sector": "điện", "family": "N1 period", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 tấn vào năm 2025.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-4eaa023bef040348-0-N2", "layer": "B-b", "company": "pow", "sector": "điện", "family": "N2 status", "claim": "Kế hoạch than dự kiến giao nhận của Công ty là 3.000.000 tấn.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-4eaa023bef040348-0-N5", "layer": "B-b", "company": "pow", "sector": "điện", "family": "N5 basis", "claim": "Công ty đặt mục tiêu Kế hoạch than dự kiến giao nhận đạt 3.000.000 tấn/tấn sản phẩm vào năm 2026.", "evidence": "Tập đoàn công nghiệp Than - Khoáng sản Việt Nam (TKV): Tổng công ty Đông Bắc - Bộ Quốc Phòng (TCTĐB): CTCP Tập đoàn Hoành Sơn (HSG): Kế hoạch than dự kiến giao nhận trong năm 2026: 3.000.000 tấn, trong đó: Nguồn khác (Công ty TNHH Năng lượng Xekong, TCTĐB, HSG,...): PV Power/ PV Power Fuel đã tích cực tìm kiếm đa dạng hoá nguồn than để luôn chủ động nguồn cung cấp phục vụ nhu cầu sản xuất.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "3.000.000", "unit_text": "tấn", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2026, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Kế hoạch than dự kiến giao nhận", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-3958feef5779ca82-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO của Công ty là 0 m3.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-3958feef5779ca82-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO của Công ty là 5 m3.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "5", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-3958feef5779ca82-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu TỔNG LƯỢNG NƯỚC ĐẦU VÀO đạt 0 m3 vào năm 2030.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-3958feef5779ca82-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO của Công ty là 0 m3/tấn sản phẩm.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-3958feef5779ca82-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 0 m3.", "evidence": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO THEO NGUỒN 0 m3 (Không áp dụng) Nước từ các tổ chức khác Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi CTCP Cấp thoát nước Cần Thơ và Công ty TNHH MTV Công trình Đô thị Hậu Giang.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "0", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "TỔNG LƯỢNG NƯỚC ĐẦU VÀO", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "claude+32b"}\n{"id": "bh-d51355abee1d0e73-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Thuốc trong nước chỉ chiếm của Công ty đạt 50%.", "evidence": "Doanh nghiệp Dược nước ngoài với công nghệ hiện đại, kinh nghiệm quốc tế, tiềm lực tài chính mạnh cùng với tâm lý chuộng hàng ngoại là những nguyên nhân mà thuốc trong nước chỉ chiếm gần 50% thị phần cả nước.", "labels": ["S"], "claim_ir": {"metric": "other_env", "value_text": "50", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thuốc trong nước chỉ chiếm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "50", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thuốc trong nước chỉ chiếm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-d51355abee1d0e73-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Thuốc trong nước chỉ chiếm của Công ty đạt 65%.", "evidence": "Doanh nghiệp Dược nước ngoài với công nghệ hiện đại, kinh nghiệm quốc tế, tiềm lực tài chính mạnh cùng với tâm lý chuộng hàng ngoại là những nguyên nhân mà thuốc trong nước chỉ chiếm gần 50% thị phần cả nước.", "labels": ["C"], "claim_ir": {"metric": "other_env", "value_text": "65", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thuốc trong nước chỉ chiếm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "50", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thuốc trong nước chỉ chiếm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n{"id": "bh-d51355abee1d0e73-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu thuốc trong nước chỉ chiếm đạt 50% vào năm 2030.", "evidence": "Doanh nghiệp Dược nước ngoài với công nghệ hiện đại, kinh nghiệm quốc tế, tiềm lực tài chính mạnh cùng với tâm lý chuộng hàng ngoại là những nguyên nhân mà thuốc trong nước chỉ chiếm gần 50% thị phần cả nước.", "labels": ["N"], "claim_ir": {"metric": "other_env", "value_text": "50", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "thuốc trong nước chỉ chiếm", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "other_env", "value_text": "50", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "thuốc trong nước chỉ chiếm", "subject_text": "", "extra": {}}, "fold": 0, "set": "heldout_v12", "pair": "32b+readerB"}\n', 'experiments/AG-1/out/g_v12/4b_cases.jsonl': '{"id": "bh-256e9fb0b2610f5e-0-S0", "label": "S", "why": ["300 kw: S |0.3 − 0.3| = 0 power_mw; dung sai 0.001"]}\n{"id": "bh-256e9fb0b2610f5e-0-C1", "label": "C", "why": ["375 kw: C |0.375 − 0.3| = 0.075 power_mw; dung sai 0.001"]}\n{"id": "bh-256e9fb0b2610f5e-0-C2", "label": "C", "why": ["300 kw: C |300 − 0.3| = 299.7 power_mw; dung sai 0.5005"]}\n{"id": "bh-256e9fb0b2610f5e-0-N2", "label": "N", "why": ["300 kw: N khác thực hiện / mục tiêu: target / actual"]}\n{"id": "bh-256e9fb0b2610f5e-0-N5", "label": "N", "why": ["300 kw/tấn sản phẩm: N khác đơn vị: power_mw/tấn sản phẩm / power_mw"]}\n{"id": "bh-256e9fb0b2610f5e-0-N6", "label": "N", "why": ["300 kw: N khác chỉ số: renewable_energy / electricity_consumption"]}\n{"id": "bh-86c0150d1bdb2265-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-86c0150d1bdb2265-0-C1", "label": "N", "why": ["7 Đơn vị: N "]}\n{"id": "bh-86c0150d1bdb2265-0-C2", "label": "N", "why": ["02 Đơn vị: N "]}\n{"id": "bh-86c0150d1bdb2265-0-N2", "label": "N", "why": ["02 : N "]}\n{"id": "bh-2924cd23d07ec79f-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-2924cd23d07ec79f-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-2924cd23d07ec79f-0-C3", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-2924cd23d07ec79f-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-12d6bfbb4289bee1-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-12d6bfbb4289bee1-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-12d6bfbb4289bee1-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-S0", "label": "N", "why": ["0,03 %: N khác loại giá trị: intensity / share"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-C1", "label": "N", "why": ["15,03 %: N khác loại giá trị: intensity / share"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-N1", "label": "N", "why": ["0,03 %: N khác loại giá trị: intensity / share"]}\n{"id": "bh-a37ff2f36e8c7a2b-0-N2", "label": "N", "why": ["0,03 %: N khác loại giá trị: intensity / share"]}\n{"id": "bh-42bb04afc41b5a1c-1-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-42bb04afc41b5a1c-1-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-42bb04afc41b5a1c-1-N1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-42bb04afc41b5a1c-1-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-42bb04afc41b5a1c-1-N6", "label": "N", "why": ["12 %: N khác loại giá trị: share / change"]}\n{"id": "bh-489f9d772e830f0e-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-489f9d772e830f0e-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-489f9d772e830f0e-0-C2", "label": "N", "why": ["950 ha: N "]}\n{"id": "bh-489f9d772e830f0e-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-489f9d772e830f0e-0-N5", "label": "N", "why": ["950 ha/tấn sản phẩm: N "]}\n{"id": "bh-2abb39872ba73557-0-S0", "label": "N", "why": ["7.000 viên: N khác đơn vị: viên / viên pin cũ"]}\n{"id": "bh-2abb39872ba73557-0-C1", "label": "N", "why": ["4.900 viên: N khác đơn vị: viên / viên pin cũ"]}\n{"id": "bh-2abb39872ba73557-0-C2", "label": "C", "why": ["7.000 viên pin cũ: C |7e+06 − 7000| = 6.993e+06 viên pin cũ; dung sai 500.5"]}\n{"id": "bh-2abb39872ba73557-0-N2", "label": "N", "why": ["7.000 viên pin cũ: N khác thực hiện / mục tiêu: target / actual"]}\n{"id": "bh-2abb39872ba73557-0-N5", "label": "N", "why": ["7.000 viên pin cũ/tấn sản phẩm: N khác loại giá trị: intensity / absolute"]}\n{"id": "bh-2abb39872ba73557-0-N6", "label": "N", "why": ["7.000 viên pin cũ: N khác chỉ số: waste_generated / waste_recycled"]}\n{"id": "bh-752eb0e07be3dea9-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-752eb0e07be3dea9-0-C1", "label": "N", "why": ["6 cây: N khác loại giá trị: absolute / intensity"]}\n{"id": "bh-752eb0e07be3dea9-0-N2", "label": "N", "why": ["1 cây: N khác loại giá trị: absolute / intensity"]}\n{"id": "bh-752eb0e07be3dea9-0-N5", "label": "N", "why": ["1 cây/tấn sản phẩm: N khác đơn vị: count_tree/tấn sản phẩm / count_tree"]}\n{"id": "bh-e4a22e9b25091836-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e4a22e9b25091836-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e4a22e9b25091836-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-079acf7a91109441-0-S0", "label": "S", "why": ["9 ha: S |9000 − 9000| = 0 area_ha; dung sai 1000"]}\n{"id": "bh-079acf7a91109441-0-C1", "label": "C", "why": ["13 ha: C |13000 − 9000| = 4000 area_ha; dung sai 1000"]}\n{"id": "bh-079acf7a91109441-0-C2", "label": "C", "why": ["9 ha: C |9e+06 − 9000| = 8.991e+06 area_ha; dung sai 500500"]}\n{"id": "bh-079acf7a91109441-0-N2", "label": "N", "why": ["9 ha: N khác loại giá trị: target / absolute"]}\n{"id": "bh-079acf7a91109441-0-N5", "label": "N", "why": ["9 ha/tấn sản phẩm: N khác loại giá trị: intensity / absolute"]}\n{"id": "bh-cdea1d4e020ecf9e-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-cdea1d4e020ecf9e-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-cdea1d4e020ecf9e-0-N1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-cdea1d4e020ecf9e-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-cdea1d4e020ecf9e-0-N5", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-24ec31c88923203e-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-24ec31c88923203e-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-24ec31c88923203e-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-7e4c135fa8258de0-0-S0", "label": "N", "why": ["860.512 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-0-S1", "label": "N", "why": ["860,5 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-0-C1", "label": "N", "why": ["602.358 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-0-C2", "label": "N", "why": ["860.512 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-0-N2", "label": "N", "why": ["860.512 KW: N khác loại giá trị: target / absolute; khác loại giá trị: target / absolute"]}\n{"id": "bh-7e4c135fa8258de0-0-N5", "label": "N", "why": ["860.512 KW/tấn sản phẩm: N khác loại giá trị: intensity / absolute; khác loại giá trị: intensity / absolute"]}\n{"id": "bh-7e4c135fa8258de0-0-N6", "label": "N", "why": ["860.512 KW: N khác chỉ số: renewable_energy / electricity_consumption; khác chỉ số: renewable_energy / electricity_consumption"]}\n{"id": "bh-7e4c135fa8258de0-1-S0", "label": "N", "why": ["241.728 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-1-S1", "label": "N", "why": ["241,7 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-1-C1", "label": "N", "why": ["193.382 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-1-C2", "label": "N", "why": ["241.728 KW: N khác kỳ: None / 2021; khác kỳ: None / 2021"]}\n{"id": "bh-7e4c135fa8258de0-1-N2", "label": "N", "why": ["241.728 KW: N khác loại giá trị: target / absolute; khác loại giá trị: target / absolute"]}\n{"id": "bh-7e4c135fa8258de0-1-N5", "label": "N", "why": ["241.728 KW/tấn sản phẩm: N khác loại giá trị: intensity / absolute; khác loại giá trị: intensity / absolute"]}\n{"id": "bh-7e4c135fa8258de0-1-N6", "label": "N", "why": ["241.728 KW: N khác chỉ số: renewable_energy / electricity_consumption; khác chỉ số: renewable_energy / electricity_consumption"]}\n{"id": "bh-2b222a6ddb74a7f1-0-S0", "label": "S", "why": ["9 ha: S |9000 − 9000| = 0 area_ha; dung sai 1000"]}\n{"id": "bh-2b222a6ddb74a7f1-0-C1", "label": "C", "why": ["14 ha: C |14000 − 9000| = 5000 area_ha; dung sai 1000"]}\n{"id": "bh-2b222a6ddb74a7f1-0-C2", "label": "C", "why": ["9 ha: C |9e+06 − 9000| = 8.991e+06 area_ha; dung sai 500500"]}\n{"id": "bh-2b222a6ddb74a7f1-0-N2", "label": "N", "why": ["9 ha: N khác loại giá trị: target / absolute"]}\n{"id": "bh-2b222a6ddb74a7f1-0-N5", "label": "N", "why": ["9 ha/tấn sản phẩm: N khác loại giá trị: intensity / absolute"]}\n{"id": "bh-aae28ea7d360fa29-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-aae28ea7d360fa29-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-aae28ea7d360fa29-0-C3", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-aae28ea7d360fa29-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-aae28ea7d360fa29-0-N6", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-aae28ea7d360fa29-1-S0", "label": "N", "why": ["2.760 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2021 / None; khác kỳ: 2021 / None; khác kỳ: 2021 / None"]}\n{"id": "bh-aae28ea7d360fa29-1-C1", "label": "N", "why": ["3.864 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2021 / None; khác kỳ: 2021 / None; khác kỳ: 2021 / None"]}\n{"id": "bh-aae28ea7d360fa29-1-C2", "label": "N", "why": ["2.760 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2021 / None; khác kỳ: 2021 / None; khác kỳ: 2021 / None"]}\n{"id": "bh-aae28ea7d360fa29-1-N1", "label": "N", "why": ["2.760 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2020 / None; khác kỳ: 2020 / None; khác kỳ: 2020 / None"]}\n{"id": "bh-aae28ea7d360fa29-1-N2", "label": "N", "why": ["2.760 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác loại giá trị: target / absolute; khác loại giá trị: target / absolute; khác loại giá trị: target / absolute"]}\n{"id": "bh-aae28ea7d360fa29-1-N5", "label": "N", "why": ["2.760 kg/tấn sản phẩm: N khác chỉ số: fuel_consumption / electricity_consumption; khác loại giá trị: intensity / absolute; khác loại giá trị: intensity / absolute; khác loại giá trị: intensity / absolute"]}\n{"id": "bh-aae28ea7d360fa29-2-S0", "label": "N", "why": ["855 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2021 / None; khác kỳ: 2021 / None; khác kỳ: 2021 / None"]}\n{"id": "bh-aae28ea7d360fa29-2-C1", "label": "N", "why": ["1.069 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2021 / None; khác kỳ: 2021 / None; khác kỳ: 2021 / None"]}\n{"id": "bh-aae28ea7d360fa29-2-C2", "label": "N", "why": ["855 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2021 / None; khác kỳ: 2021 / None; khác kỳ: 2021 / None"]}\n{"id": "bh-aae28ea7d360fa29-2-N1", "label": "N", "why": ["855 kg: N khác chỉ số: fuel_consumption / electricity_consumption; khác kỳ: 2020 / None; khác kỳ: 2020 / None; khác kỳ: 2020 / None"]}\n{"id": "bh-aae28ea7d360fa29-2-N2", "label": "N", "why": ["855 kg: N khác chỉ số: air_pollutants / electricity_consumption; khác chỉ số: air_pollutants / fuel_consumption; khác chỉ số: air_pollutants / fuel_consumption; khác chỉ số: air_pollutants / fuel_consumption"]}\n{"id": "bh-aae28ea7d360fa29-2-N5", "label": "N", "why": ["855 kg/tấn sản phẩm: N khác chỉ số: air_pollutants / electricity_consumption; khác chỉ số: air_pollutants / fuel_consumption; khác chỉ số: air_pollutants / fuel_consumption; khác chỉ số: air_pollutants / fuel_consumption"]}\n{"id": "bh-aae28ea7d360fa29-4-S0", "label": "S", "why": ["100 lít: S khác chỉ số: fuel_consumption / electricity_consumption; khác đơn vị: volume_m3 / mass_t; khác đơn vị: volume_m3 / mass_t; |0.1 − 0.1| = 0 volume_m3; dung sai 0.001"]}\n{"id": "bh-aae28ea7d360fa29-4-C1", "label": "C", "why": ["80 lít: C khác chỉ số: fuel_consumption / electricity_consumption; khác đơn vị: volume_m3 / mass_t; khác đơn vị: volume_m3 / mass_t; |0.08 − 0.1| = 0.02 volume_m3; dung sai 0.001"]}\n{"id": "bh-aae28ea7d360fa29-4-C2", "label": "C", "why": ["100 lít: C khác chỉ số: fuel_consumption / electricity_consumption; khác đơn vị: volume_m3 / mass_t; khác đơn vị: volume_m3 / mass_t; |100 − 0.1| = 99.9 volume_m3; dung sai 0.5005"]}\n{"id": "bh-aae28ea7d360fa29-4-N2", "label": "N", "why": ["100 lít: N khác chỉ số: fuel_consumption / electricity_consumption; khác loại giá trị: target / absolute; khác loại giá trị: target / absolute; khác loại giá trị: target / absolute"]}\n{"id": "bh-aae28ea7d360fa29-4-N5", "label": "N", "why": ["100 lít/tấn sản phẩm: N khác chỉ số: fuel_consumption / electricity_consumption; khác loại giá trị: intensity / absolute; khác loại giá trị: intensity / absolute; khác loại giá trị: intensity / absolute"]}\n{"id": "bh-bbd4e51bc7042cbc-0-S0", "label": "N", "why": ["83,8 %: N "]}\n{"id": "bh-bbd4e51bc7042cbc-0-C1", "label": "N", "why": ["68,8 %: N "]}\n{"id": "bh-bbd4e51bc7042cbc-0-N2", "label": "N", "why": ["83,8 %: N "]}\n{"id": "bh-bea6856b8ea1f475-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-bea6856b8ea1f475-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-bea6856b8ea1f475-0-C3", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-bea6856b8ea1f475-0-N1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-bea6856b8ea1f475-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-3c5c5f73d7747fec-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-3c5c5f73d7747fec-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-3c5c5f73d7747fec-0-C2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-3c5c5f73d7747fec-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-3c5c5f73d7747fec-0-N5", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-ac0dcc6352bdadad-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-ac0dcc6352bdadad-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-ac0dcc6352bdadad-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-8be4715d7a95430c-0-S0", "label": "S", "why": ["35 %: S |35 − 35| = 0 percent; dung sai 1"]}\n{"id": "bh-8be4715d7a95430c-0-C1", "label": "C", "why": ["50 %: C |50 − 35| = 15 percent; dung sai 1"]}\n{"id": "bh-8be4715d7a95430c-0-N1", "label": "N", "why": ["35 %: N khác kỳ: 2024 / 2025"]}\n{"id": "bh-8be4715d7a95430c-0-N2", "label": "N", "why": ["35 %: N khác thực hiện / mục tiêu: target / actual"]}\n{"id": "bh-8be4715d7a95430c-0-N6", "label": "N", "why": ["35 %: N khác chỉ số: renewable_energy / electricity_consumption"]}\n{"id": "bh-de130211938fb92a-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-de130211938fb92a-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-de130211938fb92a-0-C2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-de130211938fb92a-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-de130211938fb92a-0-N5", "label": "N", "why": ["2.300 nhân viên/tấn sản phẩm: N "]}\n{"id": "bh-e3f000701dd57fce-0-S0", "label": "N", "why": ["39,2 %: N "]}\n{"id": "bh-e3f000701dd57fce-0-C1", "label": "N", "why": ["54,2 %: N "]}\n{"id": "bh-e3f000701dd57fce-0-N2", "label": "N", "why": ["39,2 %: N "]}\n{"id": "bh-e3f000701dd57fce-1-S0", "label": "N", "why": ["43,6 %: N "]}\n{"id": "bh-e3f000701dd57fce-1-C1", "label": "N", "why": ["58,6 %: N "]}\n{"id": "bh-e3f000701dd57fce-1-N2", "label": "N", "why": ["43,6 %: N "]}\n{"id": "bh-e3f000701dd57fce-2-S0", "label": "N", "why": ["14,6 %: N "]}\n{"id": "bh-e3f000701dd57fce-2-C1", "label": "N", "why": ["29,6 %: N "]}\n{"id": "bh-e3f000701dd57fce-2-N2", "label": "N", "why": ["14,6 %: N "]}\n{"id": "bh-e3f000701dd57fce-3-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e3f000701dd57fce-3-C1", "label": "N", "why": ["10,9 %: N "]}\n{"id": "bh-e3f000701dd57fce-3-C3", "label": "N", "why": ["10,9 %: N "]}\n{"id": "bh-e3f000701dd57fce-3-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e771363ec78bac11-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e771363ec78bac11-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e771363ec78bac11-0-C2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e771363ec78bac11-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-e771363ec78bac11-0-N5", "label": "N", "why": ["651 giờ/tấn sản phẩm: N "]}\n{"id": "bh-1c6744cbbe0ddbef-0-S0", "label": "N", "why": ["5 chai: N "]}\n{"id": "bh-1c6744cbbe0ddbef-0-C1", "label": "N", "why": ["10 chai: N "]}\n{"id": "bh-1c6744cbbe0ddbef-0-C2", "label": "N", "why": ["5 chai: N "]}\n{"id": "bh-1c6744cbbe0ddbef-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-1c6744cbbe0ddbef-0-N5", "label": "N", "why": ["5 chai/tấn sản phẩm: N "]}\n{"id": "bh-4eaa023bef040348-0-S0", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-4eaa023bef040348-0-S1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-4eaa023bef040348-0-C1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-4eaa023bef040348-0-C2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-4eaa023bef040348-0-N1", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-4eaa023bef040348-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-4eaa023bef040348-0-N5", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n{"id": "bh-3958feef5779ca82-0-S0", "label": "N", "why": ["0 m3: N "]}\n{"id": "bh-3958feef5779ca82-0-C1", "label": "N", "why": ["5 m3: N "]}\n{"id": "bh-3958feef5779ca82-0-N2", "label": "N", "why": ["0 m3: N "]}\n{"id": "bh-3958feef5779ca82-0-N5", "label": "N", "why": ["0 m3/tấn sản phẩm: N "]}\n{"id": "bh-3958feef5779ca82-0-N6", "label": "N", "why": ["0 m3: N "]}\n{"id": "bh-d51355abee1d0e73-0-S0", "label": "N", "why": ["50 %: N "]}\n{"id": "bh-d51355abee1d0e73-0-C1", "label": "N", "why": ["65 %: N "]}\n{"id": "bh-d51355abee1d0e73-0-N2", "label": "N", "why": ["không đọc được số liệu trong tuyên bố"]}\n', 'src/quantum_gw/__init__.py': '', 'src/quantum_gw/providers/__init__.py': '', 'src/quantum_gw/verification/__init__.py': ''}
for path, text in B.items():
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    open(path, "w", encoding="utf-8").write(text)
print(len(B), "files written", flush=True)


In [ ]:
print("[cell] q4_K_M: pull + read set G", flush=True)
import subprocess, os
def sh(cmd, check=True):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("ollama pull qwen3:4b-instruct-2507-q4_K_M")
sh("python -u experiments/AG-2/product_reader.py benchmark/gs_bench_v1/gs_heldout_v12.jsonl out/q4_K_M --model qwen3:4b-instruct-2507-q4_K_M --ref experiments/AG-1/out/g_v12/4b_cases.jsonl")


In [ ]:
print("[cell] q8_0: pull + read set G", flush=True)
import subprocess, os
def sh(cmd, check=True):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("ollama pull qwen3:4b-instruct-2507-q8_0")
sh("python -u experiments/AG-2/product_reader.py benchmark/gs_bench_v1/gs_heldout_v12.jsonl out/q8_0 --model qwen3:4b-instruct-2507-q8_0 --ref experiments/AG-1/out/g_v12/4b_cases.jsonl")


In [ ]:
print("[cell] done", flush=True)
import subprocess
print(subprocess.run("cat out/*/summary.json", shell=True, capture_output=True, text=True).stdout)
print("ALL CELLS OK", flush=True)
